# Pipeline Lengkap: Analisis Polutan NO₂ — Kecamatan Kedungpring

**Mata Kuliah:** Proyek Sains Data — Semester 5
**Wilayah:** Kecamatan Kedungpring, Kabupaten Lamongan
**Polutan:** NO₂ (Nitrogen Dioksida)
**Rentang Waktu:** 31 Agustus 2025 — 31 Agustus 2026

## Tujuan Notebook

Notebook ini menggabungkan **seluruh alur kerja** analisis polutan NO₂ Kecamatan
Kedungpring menjadi satu file, dari data mentah hingga siap disimpan ke database:

1. **Ekstraksi Data** — menarik data Sentinel-5P L2 (band NO2) via openEO Copernicus
   Data Space untuk Kecamatan Kedungpring.
2. **Deteksi Outlier & Imputasi** — membersihkan data menggunakan metode **IQR**,
   lalu mengisi seluruh nilai kosong hingga **0 NaN tersisa**.
3. **Ekstraksi Fitur (68 fitur TSFEL)** — mengubah deret waktu NO₂ yang sudah bersih
   menjadi 68 nilai fitur (domain statistik, temporal, dan spektral).
4. **Upload ke Database Aiven** — menyimpan hasil fitur ke tabel database.

> **Catatan penting:** Bagian 4 (upload Aiven) memerlukan **informasi koneksi database
> milikmu sendiri** (host, port, nama database, kredensial, nama tabel). Notebook ini
> menyediakan strukturnya secara lengkap — kamu hanya perlu mengisi bagian yang ditandai
> `# TODO` sebelum menjalankannya.

---
# Tahap 1 — Ekstraksi Data

## 1.1 Import Pustaka

In [27]:
import os
import openeo
import xarray as xr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import inspect
import tsfel.feature_extraction.features as tsfel_features

NC_DIR = "../data/nc/"
CSV_DIR = "../data/csv/"
FEATURE_DIR = "../data/features/"
os.makedirs(NC_DIR, exist_ok=True)
os.makedirs(CSV_DIR, exist_ok=True)
os.makedirs(FEATURE_DIR, exist_ok=True)

print("Folder output siap.")

Folder output siap.


## 1.2 Koneksi & Otentikasi ke Copernicus Data Space

Menghubungkan ke server openEO menggunakan **OIDC Device Code Flow**. Saat dijalankan,
akan muncul tautan otentikasi — buka di browser, login dengan akun Copernicus Data
Space Ecosystem (CDSE), lalu proses akan otomatis melanjutkan.

In [28]:
connection = openeo.connect("openeo.dataspace.copernicus.eu")
connection.authenticate_oidc()
print("Terhubung sebagai:", connection.describe_account())

Authenticated using refresh token.
Terhubung sebagai: {'info': {'oidc_userinfo': {'email': 'triswanti1395@gmail.com', 'email_verified': True, 'family_name': "Jannatul Ma'wa", 'given_name': 'Triswanti', 'name': "Triswanti Jannatul Ma'wa", 'preferred_username': 'triswanti1395@gmail.com', 'sub': '08868660-f1d0-4cae-bac7-ad9a53fa7209'}}, 'name': "Triswanti Jannatul Ma'wa", 'user_id': '08868660-f1d0-4cae-bac7-ad9a53fa7209'}


## 1.3 Area of Interest (AOI): Kecamatan Kedungpring

Koordinat dikonversi dari data resmi *LKjIP Kecamatan Kedungpring 2024*
(112°10′01″–112°13′28″ BT, 07°08′19″–07°12′27″ LS). Luas wilayah 84,54 km², 23 desa.

> Masih berupa bounding box (persegi pembatas) — ganti dengan poligon administratif
> resmi (GADM/BIG/Kemendagri) jika dibutuhkan presisi lebih tinggi.

In [29]:
KEDUNGPRING_BBOX = {
    "west": 112.1669,
    "south": -7.2075,
    "east": 112.2244,
    "north": -7.1386,
}

KEDUNGPRING_POLYGON = {
    "type": "Polygon",
    "coordinates": [[
        [112.1669, -7.1386],
        [112.2244, -7.1386],
        [112.2244, -7.2075],
        [112.1669, -7.2075],
        [112.1669, -7.1386],
    ]]
}

TEMPORAL_EXTENT = ["2025-08-31", "2026-08-31"]
target_pollutant = "NO2"

print("AOI Kedungpring:", KEDUNGPRING_BBOX)
print("Rentang waktu:", TEMPORAL_EXTENT)

AOI Kedungpring: {'west': 112.1669, 'south': -7.2075, 'east': 112.2244, 'north': -7.1386}
Rentang waktu: ['2025-08-31', '2026-08-31']


## 1.4 Load Collection, Agregasi, & Download

- `load_collection` — memuat band NO2 dari koleksi `SENTINEL_5P_L2`.
- `aggregate_temporal_period` — rata-rata harian.
- `aggregate_spatial` — rata-rata dalam polygon Kedungpring.
- `download` — mengunduh hasil sebagai NetCDF, lalu dikonversi ke CSV.

In [31]:
# Gunakan path absolut agar tidak bergantung pada working directory saat notebook dijalankan
NC_DIR_ABS = os.path.abspath(NC_DIR)
os.makedirs(NC_DIR_ABS, exist_ok=True)

nc_path = os.path.join(NC_DIR_ABS, "no2_kedungpring.nc")
print("Path absolut tujuan unduhan:", nc_path)

try:
    spatial_result.download(nc_path, format="netCDF")
    print("Unduhan selesai:", nc_path)
except PermissionError as e:
    print("Gagal menulis file karena izin ditolak.")
    print("Detail:", e)
    print(
        "\nKemungkinan penyebab:\n"
        "1. Folder tujuan berada di lokasi yang tidak bisa ditulis (cek hasil diagnostik di atas)\n"
        "2. File sedang dibuka/terkunci aplikasi lain (tutup OneDrive sync/Explorer/Excel di folder ini)\n"
        "3. Coba jalankan VS Code sebagai Administrator, atau pindahkan folder proyek ke luar "
        "OneDrive/Documents yang tersinkronisasi cloud"
    )
    raise

Path absolut tujuan unduhan: d:\Semester 5\Proyek sain data\PSD\data\nc\no2_kedungpring.nc


Preflight process graph validation failed: ('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None))


Gagal menulis file karena izin ditolak.
Detail: [Errno 13] Permission denied

Kemungkinan penyebab:
1. Folder tujuan berada di lokasi yang tidak bisa ditulis (cek hasil diagnostik di atas)
2. File sedang dibuka/terkunci aplikasi lain (tutup OneDrive sync/Explorer/Excel di folder ini)
3. Coba jalankan VS Code sebagai Administrator, atau pindahkan folder proyek ke luar OneDrive/Documents yang tersinkronisasi cloud


PermissionError: [Errno 13] Permission denied

### Catatan: Deteksi Kolom Tanggal Secara Fleksibel

Nama kolom waktu hasil `to_dataframe()` pada `xarray` **tidak selalu** bernama `"time"`
atau `"date"` secara eksplisit — openEO terkadang memberi nama dimensi waktu hanya `"t"`
saja. Jika deteksi kolom hanya mencari kata `"time"`/`"date"`, hasilnya bisa berupa list
kosong dan menyebabkan `IndexError: list index out of range`.

Sel berikut mengatasi ini dengan mencoba beberapa nama kolom umum (`date`, `time`, `t`),
lalu sebagai jalan terakhir mencari kolom mana pun yang bertipe `datetime`. Diagnostik
(`print(df.columns.tolist())`) juga ditampilkan agar struktur data hasil `aggregate_spatial`
selalu terlihat jelas sebelum diproses lebih lanjut.

In [ ]:
ds = xr.open_dataset(nc_path)
df_raw = ds.to_dataframe().reset_index()

# --- Diagnostik: lihat dulu struktur kolom sebenarnya ---
print("Kolom yang tersedia:", df_raw.columns.tolist())
print(df_raw.dtypes)

# Deteksi kolom tanggal: coba beberapa kemungkinan nama umum dari openEO
possible_date_names = ["date", "time", "t", "time_bnds"]
date_col = None
for name in possible_date_names:
    if name in df_raw.columns:
        date_col = name
        break

# Fallback terakhir: cari kolom apapun yang bertipe datetime
if date_col is None:
    datetime_cols = [c for c in df_raw.columns if pd.api.types.is_datetime64_any_dtype(df_raw[c])]
    if datetime_cols:
        date_col = datetime_cols[0]

if date_col is None:
    raise ValueError(
        f"Tidak ditemukan kolom tanggal. Kolom yang ada: {df_raw.columns.tolist()}. "
        "Cek output 'Kolom yang tersedia' di atas untuk menentukan nama kolom waktu yang benar."
    )

print(f"\nKolom tanggal terdeteksi: '{date_col}'")

# Kolom nilai NO2: ambil kolom numerik selain kolom tanggal
value_candidates = [c for c in df_raw.columns if c != date_col and pd.api.types.is_numeric_dtype(df_raw[c])]
if not value_candidates:
    raise ValueError(f"Tidak ditemukan kolom nilai numerik. Kolom yang ada: {df_raw.columns.tolist()}")
value_col = value_candidates[-1]
print(f"Kolom nilai terdeteksi: '{value_col}'")

df_raw = df_raw[[date_col, value_col]].rename(columns={date_col: "date", value_col: "NO2"})
df_raw["date"] = pd.to_datetime(df_raw["date"])

raw_csv_path = os.path.join(CSV_DIR, "NO2-Kedungpring.csv")
df_raw.to_csv(raw_csv_path, index=False)
print(f"\nCSV mentah disimpan: {raw_csv_path} ({len(df_raw)} baris)")
df_raw.head()

---
# Tahap 2 — Deteksi Outlier (IQR) & Imputasi Missing Value

## 2.1 Konversi Tipe Numerik

Kolom NO₂ dipaksa menjadi tipe numerik; nilai yang gagal dikonversi otomatis menjadi
`NaN` sehingga ikut ditangani pada tahap imputasi.

In [ ]:
df = df_raw.sort_values("date").reset_index(drop=True).copy()
df[target_pollutant] = pd.to_numeric(df[target_pollutant], errors="coerce")

n_missing_before = df[target_pollutant].isna().sum()
print(f"Jumlah baris                                   : {len(df)}")
print(f"Nilai non-numerik/kosong (dikonversi jadi NaN) : {n_missing_before}")

## 2.2 Deteksi Outlier — Metode IQR

Outlier dideteksi berdasarkan rentang $[Q1 - 1.5 \times IQR,\; Q3 + 1.5 \times IQR]$,
lalu ditandai sebagai NaN (bukan dihapus barisnya) agar struktur deret waktu tetap
utuh.

In [ ]:
Q1 = df[target_pollutant].quantile(0.25)
Q3 = df[target_pollutant].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

is_outlier = (df[target_pollutant] < lower_bound) | (df[target_pollutant] > upper_bound)
n_outlier = is_outlier.sum()

print(f"Q1 = {Q1:.6f} | Q3 = {Q3:.6f} | IQR = {IQR:.6f}")
print(f"Batas bawah = {lower_bound:.6f} | Batas atas = {upper_bound:.6f}")
print(f"Jumlah outlier terdeteksi: {n_outlier} dari {df[target_pollutant].notna().sum()} data valid")

df_before_impute = df.copy()
df_before_impute["is_outlier"] = is_outlier

df.loc[is_outlier, target_pollutant] = np.nan

In [ ]:
# Visualisasi: normal vs outlier
normal = df_before_impute[~df_before_impute["is_outlier"]]
outliers = df_before_impute[df_before_impute["is_outlier"]]

plt.figure(figsize=(12, 4))
plt.scatter(normal["date"], normal[target_pollutant], color="steelblue", s=20, label="Normal")
plt.scatter(outliers["date"], outliers[target_pollutant], color="crimson", s=45, marker="x", label="Outlier (IQR)")
plt.title(f"Deteksi Outlier {target_pollutant} — Kecamatan Kedungpring (Metode IQR)")
plt.xlabel("Tanggal")
plt.ylabel(f"Konsentrasi {target_pollutant}")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2.3 Imputasi hingga 0 NaN

Missing value asli dan outlier yang baru ditandai diisi bersamaan menggunakan
interpolasi linear berbasis waktu, dilanjutkan forward/backward fill. Sebuah `assert`
memastikan **tidak ada NaN tersisa** sebelum lanjut ke tahap ekstraksi fitur.

In [ ]:
df_clean = df.set_index("date").interpolate(method="time").ffill().bfill()

n_missing_after = df_clean[target_pollutant].isna().sum()
print(f"Missing value SEBELUM imputasi : {df[target_pollutant].isna().sum()}")
print(f"Missing value SETELAH imputasi : {n_missing_after}")

assert n_missing_after == 0, "Masih ada NaN tersisa setelah imputasi!"
print("\n✅ Data NO2 Kedungpring sudah 100% bersih — 0 NaN, 0 outlier tersisa.")

df_clean = df_clean.reset_index()

clean_csv_path = os.path.join(CSV_DIR, "NO2-Kedungpring-clean.csv")
df_clean.to_csv(clean_csv_path, index=False)
print(f"Data bersih disimpan: {clean_csv_path}")

df_clean.head()

---
# Tahap 3 — Ekstraksi 68 Fitur dengan TSFEL

## 3.1 Menyiapkan Sinyal

Kolom NO₂ yang sudah bersih (hasil Tahap 2) diambil sebagai array 1 dimensi
(`signal_1d`), dengan `fs=1` (frekuensi sampling harian — satu nilai per hari, sesuai
hasil agregasi temporal pada Tahap 1).

In [ ]:
fs = 1
signal_1d = df_clean[target_pollutant].astype(float).values

print(f"Panjang sinyal: {len(signal_1d)} titik data (hari)")

## 3.2 Daftar 68 Fitur TSFEL

Daftar fungsi fitur berikut diambil langsung dari modul
`tsfel.feature_extraction.features`, mencakup tiga domain:

- **Statistical** — mis. `calc_mean`, `calc_std`, `skewness`, `kurtosis`, `interq_range`
- **Temporal** — mis. `autocorr`, `slope`, `zero_cross`, `mean_abs_diff`, `entropy`
- **Spectral** — mis. `spectral_entropy`, `fundamental_frequency`, `spectral_centroid`,
  `wavelet_energy`

In [ ]:
FEATURE_LIST = """abs_energy auc autocorr average_power calc_centroid calc_max calc_mean
calc_median calc_min calc_std calc_var dfa distance ecdf ecdf_percentile ecdf_percentile_count
ecdf_slope entropy fundamental_frequency higuchi_fractal_dimension hist_mode human_range_energy
hurst_exponent interq_range kurtosis lempel_ziv lpcc max_frequency max_power_spectrum
maximum_fractal_length mean_abs_deviation mean_abs_diff mean_diff median_abs_deviation
median_abs_diff median_diff median_frequency mfcc mse negative_turning neighbourhood_peaks
petrosian_fractal_dimension pk_pk_distance positive_turning power_bandwidth rms skewness slope
spectral_centroid spectral_decrease spectral_distance spectral_entropy spectral_kurtosis
spectral_positive_turning spectral_roll_off spectral_roll_on spectral_skewness spectral_slope
spectral_spread spectral_variation spectrogram_mean_coeff sum_abs_diff wavelet_abs_mean
wavelet_energy wavelet_entropy wavelet_std wavelet_var zero_cross""".split()

print("Jumlah fitur yang akan diekstrak:", len(FEATURE_LIST))

## 3.3 Fungsi Ekstraksi

`extract_one()` memanggil setiap fungsi fitur TSFEL secara dinamis (mengecek apakah
fungsi tersebut membutuhkan parameter `fs` atau tidak), lalu `to_scalar()` memastikan
hasilnya berupa satu angka tunggal (beberapa fungsi TSFEL mengembalikan array/dict,
mis. `mfcc` atau `lpcc`, sehingga dirata-ratakan menjadi satu nilai representatif).

In [ ]:
def to_scalar(result):
    if isinstance(result, dict) and "values" in result:
        result = result["values"]
    if isinstance(result, (list, tuple, np.ndarray)):
        arr = np.asarray(result, dtype=float)
        return float(np.nanmean(arr))
    return float(result)


def extract_one(fn_name, signal, fs):
    fn = getattr(tsfel_features, fn_name)
    params = inspect.signature(fn).parameters
    if "fs" in params:
        result = fn(signal, fs)
    else:
        result = fn(signal)
    return to_scalar(result)


row = {}
for fn_name in FEATURE_LIST:
    try:
        row[fn_name] = extract_one(fn_name, signal_1d, fs)
    except Exception as e:
        print(f"[PERINGATAN] Gagal mengekstrak fitur '{fn_name}': {e}")
        row[fn_name] = np.nan

extracted_features_final = pd.DataFrame([row])

print(f"\nBerhasil! Jumlah fitur yang dihasilkan untuk {target_pollutant}: "
      f"{extracted_features_final.shape[1]}")
extracted_features_final

## 3.4 Menyimpan Hasil Fitur

In [ ]:
feature_csv_path = os.path.join(FEATURE_DIR, f"{target_pollutant}_Kedungpring_TSFEL.csv")
extracted_features_final.to_csv(feature_csv_path, index=False)
print(f"Fitur tersimpan: {feature_csv_path}")

---
# Tahap 4 — Upload ke Database Aiven

## 4.1 Konfigurasi Koneksi Database

> ⚠️ **Bagian ini perlu kamu lengkapi sendiri** dengan informasi koneksi Aiven-mu.
> Jangan pernah menuliskan password langsung di notebook yang akan di-*commit* ke
> GitHub — gunakan **environment variable** atau file `.env` (lihat catatan di bawah).

Install driver database yang sesuai (pilih salah satu sesuai jenis database Aiven-mu):

```bash
pip install psycopg2-binary   # jika PostgreSQL
pip install pymysql           # jika MySQL
pip install sqlalchemy
```

In [ ]:
import os
from sqlalchemy import create_engine, text

# TODO: isi sesuai info koneksi Aiven kamu (Overview -> Connection information)
DB_TYPE = "postgresql"   # atau "mysql", sesuaikan dengan jenis service Aiven-mu
DB_HOST = os.environ.get("AIVEN_DB_HOST", "TODO_ISI_HOST")
DB_PORT = os.environ.get("AIVEN_DB_PORT", "TODO_ISI_PORT")
DB_NAME = os.environ.get("AIVEN_DB_NAME", "TODO_ISI_NAMA_DATABASE")
DB_USER = os.environ.get("AIVEN_DB_USER", "TODO_ISI_USERNAME")
DB_PASSWORD = os.environ.get("AIVEN_DB_PASSWORD", "TODO_ISI_PASSWORD")
DB_TABLE = "fitur_no2_kedungpring"  # TODO: sesuaikan nama tabel tujuan

# Aiven umumnya mewajibkan SSL
if DB_TYPE == "postgresql":
    conn_str = f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}?sslmode=require"
elif DB_TYPE == "mysql":
    conn_str = f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}?ssl_verify_cert=true"
else:
    raise ValueError("DB_TYPE harus 'postgresql' atau 'mysql'")

print("Konfigurasi koneksi disiapkan (belum tersambung).")
print("Tabel tujuan:", DB_TABLE)

## 4.2 Menyiapkan Data untuk Diunggah

Menambahkan kolom metadata (`kecamatan`, `polutan`, `tanggal_ekstraksi`) pada tabel
fitur, agar data yang tersimpan di database tetap bisa ditelusuri asalnya jika nanti
digabung dengan data kecamatan lain.

In [ ]:
from datetime import datetime

upload_df = extracted_features_final.copy()
upload_df.insert(0, "kecamatan", "Kedungpring")
upload_df.insert(1, "polutan", target_pollutant)
upload_df.insert(2, "tanggal_ekstraksi", datetime.now().strftime("%Y-%m-%d"))

upload_df.head()

## 4.3 Koneksi & Insert ke Tabel Aiven

Jika tabel tujuan **belum ada**, `to_sql(..., if_exists="replace")` akan membuat tabel
baru secara otomatis mengikuti struktur `upload_df`. Jika tabel **sudah ada** dan hanya
ingin menambah baris baru, ganti `if_exists="replace"` menjadi `if_exists="append"`.

In [ ]:
try:
    engine = create_engine(conn_str)

    with engine.connect() as conn:
        result = conn.execute(text("SELECT 1"))
        print("Koneksi ke database Aiven berhasil.")

    # TODO: ganti "replace" -> "append" setelah tabel pertama kali dibuat,
    # supaya proses selanjutnya menambah baris baru, bukan menimpa tabel.
    upload_df.to_sql(DB_TABLE, engine, if_exists="replace", index=False)

    print(f"Berhasil mengunggah {len(upload_df)} baris ke tabel '{DB_TABLE}'.")

except Exception as e:
    print("Gagal terhubung/mengunggah ke database Aiven.")
    print("Detail error:", e)
    print("\nPeriksa kembali: host, port, username, password, nama database, "
          "dan apakah IP kamu sudah diizinkan di pengaturan firewall Aiven.")

## 4.4 Verifikasi Data di Database

Membaca kembali beberapa baris dari tabel untuk memastikan data benar-benar tersimpan
dengan struktur yang sesuai.

In [ ]:
try:
    verify_df = pd.read_sql(f"SELECT * FROM {DB_TABLE} LIMIT 5", engine)
    print("Verifikasi data di tabel Aiven:")
    display(verify_df)
except Exception as e:
    print("Belum bisa memverifikasi (kemungkinan koneksi belum berhasil di atas):", e)

---
# Ringkasan Pipeline

| Tahap | Proses | Output |
| ----- | ------ | ------ |
| 1. Ekstraksi | Tarik data Sentinel-5P NO₂, AOI Kecamatan Kedungpring, 31 Agu 2025–31 Agu 2026 | `NO2-Kedungpring.csv` |
| 2. Outlier & Imputasi | Deteksi outlier (IQR), imputasi hingga 0 NaN | `NO2-Kedungpring-clean.csv` |
| 3. Ekstraksi Fitur | 68 fitur TSFEL (statistical, temporal, spectral) | `NO2_Kedungpring_TSFEL.csv` |
| 4. Upload Database | Insert hasil fitur ke tabel Aiven | Tabel `fitur_no2_kedungpring` |

## Yang Perlu Kamu Lengkapi

1. **Bagian 4.1** — isi `DB_HOST`, `DB_PORT`, `DB_NAME`, `DB_USER`, `DB_PASSWORD` sesuai
   info dari dashboard Aiven (menu **Overview** → **Connection information**).
2. Sebaiknya simpan kredensial di **environment variable** atau file `.env` (jangan
   ditulis langsung di notebook), terutama jika notebook ini akan di-*push* ke GitHub
   untuk ditampilkan di web statis — supaya password tidak ikut ter-*publish*.
3. Setelah tabel berhasil dibuat pertama kali, ganti `if_exists="replace"` menjadi
   `if_exists="append"` di Bagian 4.3 supaya proses berikutnya menambah baris, bukan
   menimpa data lama.